In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib
matplotlib.use('Agg')

FIG_DIR = PROJECT_ROOT / 'outputs' / 'figures' / 'regression'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)

Project root: /home/taqis/Pone/CSCI323_Project


In [2]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data.preprocess import get_preprocessed_data
from src.models.regression import train_regression, predict_expense

In [3]:
data = get_preprocessed_data(str(PROJECT_ROOT / "data" / "raw" / "medical_insurance.csv"))

print("X_train shape:", data["X_train"].shape)
print("X_val shape:", data["X_val"].shape)

X_train shape: (1070, 9)
X_val shape: (134, 9)


In [4]:
results = train_regression(
    data["X_train"],
    data["y_train"],
    data["X_val"],
    data["y_val"],
)

print("Best Model:", results["best_model_name"])

Best Model: Linear Regression


In [5]:
results["comparison_table"]

,Model,Train RMSE,Train MAE,Train R²,Val RMSE,Val MAE,Val R²
0,Linear Regression,6105.245135,4207.976585,0.741751,5572.667818,4042.016644,0.782227
1,Ridge,6105.493012,4217.235290,0.741730,5580.997516,4057.014503,0.781575
2,Lasso,6105.247735,4208.046260,0.741751,5573.360816,4042.785959,0.782173


In [6]:
results["train_metrics"]

{'rmse': 6105.245134876944, 'mae': 4207.976585046026, 'r2': 0.7417509671301924}

In [7]:
results["val_metrics"]

{'rmse': 5572.667817842555, 'mae': 4042.016643897426, 'r2': 0.7822269868844826}

In [8]:
best_model = results["best_model"]

predictions = best_model.predict(data["X_val"])

In [9]:
plt.figure(figsize=(8, 6))
plt.scatter(data["y_val"], predictions, alpha=0.6)
plt.xlabel("Actual Expenses")
plt.ylabel("Predicted Expenses")
plt.title("Predicted vs Actual Expenses")
plt.tight_layout()
plt.savefig(FIG_DIR / "Predicted vs Actual Expenses.png", dpi=150)
plt.show()
print("Plot saved.")

Plot saved.


/tmp/ipykernel_426001/2853543250.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
residuals = data["y_val"] - predictions

plt.figure(figsize=(8, 6))
plt.hist(residuals, bins=25, edgecolor="white")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Residual Distribution")
plt.tight_layout()
plt.savefig(FIG_DIR / "Residual Distribution.png", dpi=150)
plt.show()
print("Plot saved.")

Plot saved.


/tmp/ipykernel_426001/539222616.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
coefficients = pd.DataFrame({
    "Feature": data["feature_names"],
    "Coefficient": best_model.coef_
})

coefficients["Abs"] = coefficients["Coefficient"].abs()
coefficients = coefficients.sort_values("Abs", ascending=False)
coefficients

,Feature,Coefficient,Abs
4,discount_eligibility,23650.312302,23650.312302
0,age,3614.697633,3614.697633
1,bmi,2037.268555,2037.268555
2,children,517.330947,517.330947
5,region_northeast,459.563943,459.563943
8,region_southwest,-349.665935,349.665935
7,region_southeast,-199.148439,199.148439
6,region_northwest,89.250432,89.250432
3,gender,-18.519741,18.519741


In [12]:
plt.figure(figsize=(10, 6))
plt.bar(coefficients["Feature"], coefficients["Coefficient"])
plt.xticks(rotation=45, ha="right")
plt.title("Feature Importance")
plt.tight_layout()
plt.savefig(FIG_DIR / "Feature Importance.png", dpi=150)
plt.show()
print("Plot saved.")

Plot saved.


/tmp/ipykernel_426001/2364500300.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
sample_customer = data["X_val"].iloc[[0]]

predicted_expense = predict_expense(best_model, sample_customer)

print("Predicted Expense:")
print(predicted_expense)

Predicted Expense:
4343.389050529837
